In [ ]:
# Cell 1: Interactive Prediction UI with Smart Features
import joblib
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. LOAD THE SAVED MODELS
# ==========================================
print("Loading models...")

# Classification Model
class_model = joblib.load("Class_model_LGBM_Classifier_Damage_state.pkl")

# Regression Models
reg_model_max = joblib.load("Reg_model_LGBM_Regressor_max_displacement_at_slab_center.pkl")
reg_model_res = joblib.load("Reg_model_LGBM_Regressor_slab_center_displacement_at_the_last_time_step_residual.pkl")

print("✅ Models loaded successfully!")

# ==========================================
# 2. EXTRACT FEATURE NAMES & RANGES
# ==========================================
# We use the test dataset to grab the exact column names and data distributions
df_template = pd.read_csv("dataset_test.csv")

# Remove target columns to leave only the features
targets_to_drop = [
    'Damage state', 
    'max displacement at slab center', 
    'slab center displacement at the last time step (residual)'
]
features = [col for col in df_template.columns if col not in targets_to_drop]

# ==========================================
# 3. BUILD THE INTERACTIVE UI
# ==========================================
input_widgets = {}
ui_rows = [] # This will hold the neatly arranged display rows

for feat in features:
    # Find unique values to determine feature type
    unique_vals = df_template[feat].dropna().unique()
    num_unique = len(unique_vals)
    
    if num_unique <= 5:
        # Categorical/Discrete Features -> Dropdown selection
        sorted_vals = sorted(unique_vals.tolist())
        widget = widgets.Dropdown(
            options=sorted_vals, 
            value=sorted_vals[0],
            description=feat + ':', 
            style={'description_width': '150px'},
            layout=widgets.Layout(width='300px')
        )
        input_widgets[feat] = widget
        ui_rows.append(widget)
        
    else:
        # Continuous Features -> Numeric input with range label next to it
        feat_min = float(df_template[feat].min())
        feat_max = float(df_template[feat].max())
        feat_mean = float(df_template[feat].mean())
        
        # Enforce bounds directly in the input box to prevent bad inputs
        widget = widgets.BoundedFloatText(
            value=feat_mean,
            min=feat_min,
            max=feat_max,
            description=feat + ':', 
            style={'description_width': '150px'},
            layout=widgets.Layout(width='300px')
        )
        input_widgets[feat] = widget
        
        # Create a label showing the allowed range
        range_label = widgets.Label(
            value=f" ⬅ Range: [ {feat_min:.4f}  to  {feat_max:.4f} ]",
            style={'color': 'gray'}
        )
        
        # Group the input box and the range label side-by-side
        row = widgets.HBox([widget, range_label])
        ui_rows.append(row)

# Create a button to trigger prediction
predict_button = widgets.Button(
    description='Predict Fire Damage & Displacement',
    button_style='success',
    tooltip='Click to run inference',
    icon='check',
    layout=widgets.Layout(width='300px', margin='15px 0px 10px 0px')
)

# Output area for results
output_area = widgets.Output()

# ==========================================
# 4. DEFINE PREDICTION LOGIC
# ==========================================
def on_predict_clicked(b):
    with output_area:
        clear_output()
        
        # Gather user inputs into a dictionary, then convert to DataFrame
        user_input_data = {feat: widget.value for feat, widget in input_widgets.items()}
        input_df = pd.DataFrame([user_input_data])
        
        # --- Classification Prediction ---
        ds_pred = class_model.predict(input_df)[0]
        
        # --- Regression Predictions ---
        # Note: The regression script multiplied the targets by -1 before training. 
        # We multiply by -1 again here to return the values to their original physical sign.
        max_disp_pred = reg_model_max.predict(input_df)[0] * -1
        res_disp_pred = reg_model_res.predict(input_df)[0] * -1
        
        # --- Display Results ---
        print("="*55)
        print("🎯 PREDICTION RESULTS")
        print("="*55)
        print(f"🔥 Predicted Damage State (Encoded):  {ds_pred}")
        print(f"📉 Max Displacement at Slab Center:   {max_disp_pred:.4f}")
        print(f"📉 Residual Displacement (Last Step): {res_disp_pred:.4f}")
        print("="*55)

# Attach the function to the button
predict_button.on_click(on_predict_clicked)

# ==========================================
# 5. DISPLAY THE INTERFACE
# ==========================================
# Arrange widgets in a vertical box
ui_layout = widgets.VBox([
    widgets.HTML("<h2>🔥 Structure Fire Performance Predictor</h2>"),
    widgets.HTML("<p><i>Select values from the dropdowns and enter numbers within the specified ranges.</i></p>"),
    *ui_rows, 
    predict_button, 
    output_area
])

display(ui_layout)

Loading models...
✅ Models loaded successfully!


In [2]:
import pandas as pd
import numpy as np

# 1. Load both datasets to get the full range of values
df_train = pd.read_csv("dataset_train.csv")
df_test = pd.read_csv("dataset_test.csv")

# Combine them to see the full scope of the data
df_all = pd.concat([df_train, df_test], axis=0)

# Drop target columns if they are present
targets = [
    'Damage state', 
    'max displacement at slab center', 
    'slab center displacement at the last time step (residual)'
]
features = [col for col in df_all.columns if col not in targets]

print("📊 FEATURE ANALYSIS FOR UI DESIGN 📊")
print("="*60)

categorical_candidates = {}
continuous_candidates = {}

# 2. Analyze each feature
for feat in features:
    # Get unique values, dropping any NaNs
    unique_vals = df_all[feat].dropna().unique()
    num_unique = len(unique_vals)
    
    # If the feature has 5 or fewer unique values, treat it as a multiple choice candidate
    if num_unique <= 5:
        # Sort them if they are numeric
        if pd.api.types.is_numeric_dtype(df_all[feat]):
            unique_vals = np.sort(unique_vals)
        categorical_candidates[feat] = unique_vals.tolist()
    else:
        # Otherwise, treat it as continuous and find the min/max
        feat_min = df_all[feat].min()
        feat_max = df_all[feat].max()
        continuous_candidates[feat] = (feat_min, feat_max)

# 3. Print the results clearly
print("\n📌 FEATURES WITH <= 5 UNIQUE VALUES (Use Dropdowns):")
print("-" * 60)
for feat, vals in categorical_candidates.items():
    print(f"🔹 {feat} ({len(vals)} unique values): {vals}")

print("\n📈 CONTINUOUS FEATURES (Use Numeric Inputs/Sliders):")
print("-" * 60)
for feat, (f_min, f_max) in continuous_candidates.items():
    # Format the min and max for cleaner reading
    print(f"🔸 {feat}: Range [ {f_min:.4f}  to  {f_max:.4f} ]")

📊 FEATURE ANALYSIS FOR UI DESIGN 📊

📌 FEATURES WITH <= 5 UNIQUE VALUES (Use Dropdowns):
------------------------------------------------------------
🔹 A_s (3 unique values): [142.0, 230.0, 345.0]
🔹 AP (3 unique values): [1.0, 1.5, 3.0]
🔹 reX (2 unique values): [0.0, 1.0]
🔹 reY (2 unique values): [0.0, 1.0]

📈 CONTINUOUS FEATURES (Use Numeric Inputs/Sliders):
------------------------------------------------------------
🔸 Fire load: Range [ 100.0000  to  2100.0000 ]
🔸 Opening factor: Range [ 0.0152  to  1.0000 ]
🔸 Conductivity: Range [ -2.6289  to  1.8415 ]
🔸 Specific heat: Range [ -1.2999  to  2.6033 ]
🔸 Density: Range [ -1.3525  to  2.0069 ]
🔸 Yield stress: Range [ -2.5853  to  1.9488 ]
